# FinQA diagnosis: GPU runbook (Colab / Kaggle)
Runtime -> Change runtime type -> GPU (T4 is enough with `roberta-base`).
Run the cells in order. Each training cell saves into `runs/`, and the last cell zips everything so you can download it.
**Back-planned schedule to the Nov 8 midterm:** day 1 setup + tests + retriever (~1-2 h on T4); day 2 generator with released retrieval; day 3 generator with my retriever + taxonomy; then paste the numbers into `report/milestone.tex`.

## 0. Get the code
Upload `finqa-diagnosis.zip` (Colab: Files panel; Kaggle: add as a dataset) and unzip.

In [ ]:
!unzip -q -o finqa-diagnosis.zip -d . && cd finqa-diagnosis && ls
%cd finqa-diagnosis
!pip -q install transformers tokenizers tqdm matplotlib pytest
import torch; print(torch.__version__, torch.cuda.is_available())

## 1. Data + sanity checks (CPU, minutes)

In [ ]:
!bash scripts/00_download_data.sh
!python -m pytest -q
!python scripts/01_validate_dsl.py

## 2. Quick GPU smoke test (do this before the long runs)
`--limit` uses only N examples per split; it should finish in a couple of minutes and the generator's training loss should fall.

In [ ]:
!python scripts/05_train_retriever.py --limit 64 --epochs 1 --out runs/smoke_ret --rank_splits dev
!python scripts/06_train_generator.py --limit 64 --epochs 2 --retrieval released --out runs/smoke_gen

## 3. Generator with the released retrieval (first real baseline)

In [ ]:
!python scripts/06_train_generator.py --model roberta-base --retrieval released --out runs/gen_released

## 4. My retriever, then generator on its rankings

In [ ]:
!python scripts/05_train_retriever.py --model roberta-base --out runs/retriever
!python scripts/06_train_generator.py --model roberta-base --retrieval runs/retriever --out runs/gen_mine

## 5. Taxonomy reports

In [ ]:
!python scripts/07_taxonomy_report.py --pred runs/gen_released/pred_test.json --oracle runs/gen_released/pred_test_oracle.json --name released
!python scripts/07_taxonomy_report.py --pred runs/gen_mine/pred_test.json --oracle runs/gen_mine/pred_test_oracle.json --name mine

## 6. Package results
Download `results_gpu.zip` and send it back / drop into the repo's `results/`.

In [ ]:
!zip -qr results_gpu.zip runs results -x '*.pt'  # checkpoints excluded (large)